# Command-line interface and agent infrastructure for CogitareLink
> Tools and infrastructure for building agentic systems leveraging Cogitarelink's vocabulary management

In [ ]:
#| default_exp cli.cli

In [ ]:
#| hide
from __future__ import annotations
import sys
import json
import argparse
from typing import Dict, List, Any, Callable, Optional, Union
from pathlib import Path
import importlib.metadata

In [ ]:
#| exporti
__all__ = ['Agent', 'AgentContext', 'ToolRegistry', 'cli_main']

# Setting up the environment

We're creating a framework for agentic systems that leverage CogitareLink's vocabulary management capabilities. The agent system will follow these design principles:

1. **Tool-Based Architecture** - Agents work through well-defined tools that encapsulate functionality
2. **Compatibility with CogitareLink** - Leverage existing registry, caching, and retrieval systems
3. **Progressive Development** - Start simple and incrementally add capabilities
4. **Extensibility** - Allow users to easily add custom tools

First, we'll need to set up the core infrastructure with imports from CogitareLink:

In [ ]:
#| export
from cogitarelink.core.debug import get_logger
from cogitarelink.core.cache import InMemoryCache, DiskCache

log = get_logger("cli")
_tool_cache = InMemoryCache(maxsize=128)

# Tool Registry

The foundation of our agent system is a tool registry that manages the available tools, their metadata, and their execution. This registry allows agents to discover, validate, and execute tools in a consistent way.

In [ ]:
#| export
class ToolRegistry:
    """Registry for agent tools with metadata and validation."""
    
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}
        
    def register(self, name: str = None, description: str = None):
        """Decorator to register a function as a tool."""
        def decorator(func):
            tool_name = name or func.__name__
            tool_desc = description or func.__doc__
            
            self.tools[tool_name] = {
                "name": tool_name,
                "description": tool_desc,
                "function": func,
                "signature": getattr(func, "__annotations__", {})
            }
            return func
        return decorator
    
    def get(self, name: str) -> Dict[str, Any]:
        """Get a tool by name."""
        if name not in self.tools:
            raise ValueError(f"Tool '{name}' not found in registry")
        return self.tools[name]
    
    def list_tools(self) -> List[Dict[str, Any]]:
        """List all available tools with metadata."""
        return [
            {
                "name": name,
                "description": tool["description"],
                "signature": {k: str(v) for k, v in tool["signature"].items() if k != "return"}
            }
            for name, tool in self.tools.items()
        ]
    
    def execute(self, name: str, **kwargs) -> Any:
        """Execute a tool by name with the given arguments."""
        tool = self.get(name)
        func = tool["function"]
        return func(**kwargs)

# Testing the Tool Registry

Let's test our tool registry with a simple example:

In [ ]:
registry = ToolRegistry()

@registry.register(name="add_numbers", description="Add two numbers together")
def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

# List all tools
registry.list_tools()

[{'name': 'add_numbers',
  'description': 'Add two numbers together',
  'signature': {'a': 'int', 'b': 'int'}}]

In [ ]:
# Execute the tool
registry.execute("add_numbers", a=5, b=3)

8

# Agent Context

Next, we'll create a context for our agents to maintain state between tool executions. This context includes:

1. A memory system for storing and retrieving data
2. A history of actions for tracking tool usage
3. Integration with the caching system

In [ ]:
#| export
class AgentContext:
    """Context for agent execution with state management."""
    
    def __init__(self, agent, disk_cache: bool = True):
        self.agent = agent
        self.memory: Dict[str, Any] = {}
        self.history: List[Dict[str, Any]] = []
        
        # Set up caching
        if disk_cache:
            cache_dir = Path.home() / ".cogitarelink" / "agent_cache"
            cache_dir.mkdir(parents=True, exist_ok=True)
            self.cache = DiskCache(str(cache_dir))
        else:
            self.cache = InMemoryCache(maxsize=256)
    
    def remember(self, key: str, value: Any):
        """Store a value in agent memory."""
        self.memory[key] = value
        return value
    
    def recall(self, key: str, default: Any = None) -> Any:
        """Retrieve a value from agent memory."""
        return self.memory.get(key, default)
    
    def log_action(self, tool: str, inputs: Dict[str, Any], result: Any):
        """Log an action to the history."""
        entry = {
            "timestamp": __import__("time").time(),
            "tool": tool,
            "inputs": inputs,
            "success": result is not None and not isinstance(result, Exception),
            "result_type": type(result).__name__
        }
        self.history.append(entry)
        return entry
    
    @_tool_cache.memoize("agent")
    def cached_execute(self, tool: str, **kwargs) -> Any:
        """Execute a tool with caching."""
        return self.agent.tools.execute(tool, **kwargs)

# Agent Class

Now we'll create the main Agent class that ties everything together. This class will:

1. Maintain a tool registry
2. Create and manage its execution context
3. Register core tools that all agents should have
4. Provide methods for running tools with appropriate logging

In [ ]:
#| export
class Agent:
    """Base agent class for CogitareLink."""
    
    def __init__(self, name: str = "cogitarelink-agent"):
        self.name = name
        self.tools = ToolRegistry()
        self.context = AgentContext(self)
        self._register_core_tools()
    
    def _register_core_tools(self):
        """Register core tools available to all agents."""
        
        @self.tools.register(name="get_version", 
                           description="Get the CogitareLink version")
        def get_version() -> str:
            try:
                return importlib.metadata.version("cogitarelink")
            except importlib.metadata.PackageNotFoundError:
                return "development"
        
        @self.tools.register(name="list_tools",
                           description="List all available tools")
        def list_tools() -> List[Dict[str, Any]]:
            return self.tools.list_tools()
        
        @self.tools.register(name="get_agent_memory",
                           description="Get all items in agent memory")
        def get_agent_memory() -> Dict[str, Any]:
            return self.context.memory.copy()
    
    def register_tool(self, func: Callable, name: str = None, description: str = None):
        """Register a new tool with the agent."""
        return self.tools.register(name, description)(func)
    
    def run_tool(self, name: str, **kwargs) -> Any:
        """Run a tool and log the action."""
        try:
            result = self.tools.execute(name, **kwargs)
            self.context.log_action(name, kwargs, result)
            return result
        except Exception as e:
            self.context.log_action(name, kwargs, e)
            raise
    
    def run_cached_tool(self, name: str, **kwargs) -> Any:
        """Run a tool with caching and log the action."""
        try:
            result = self.context.cached_execute(name, **kwargs)
            self.context.log_action(name, kwargs, result)
            return result
        except Exception as e:
            self.context.log_action(name, kwargs, e)
            raise

# Testing the Agent

Let's create a simple agent to test our implementation:

In [ ]:
agent = Agent(name="test-agent")

# Register a custom tool
@agent.register_tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

# List all available tools
agent.run_tool("list_tools")

[{'name': 'get_version',
  'description': 'Get the CogitareLink version',
  'signature': {}},
 {'name': 'list_tools',
  'description': 'List all available tools',
  'signature': {}},
 {'name': 'get_agent_memory',
  'description': 'Get all items in agent memory',
  'signature': {}},
 {'name': 'multiply',
  'description': 'Multiply two numbers together.',
  'signature': {'a': 'int', 'b': 'int'}}]

In [ ]:
# Run tools and examine memory
agent.run_tool("multiply", a=4, b=5)
agent.context.remember("result", 20)
agent.context.remember("calculation", "multiplication")
agent.run_tool("get_agent_memory")

{'result': 20, 'calculation': 'multiplication'}

In [ ]:
# Check the action history
agent.context.history

[{'timestamp': 1745926381.385365,
  'tool': 'list_tools',
  'inputs': {},
  'success': True,
  'result_type': 'list'},
 {'timestamp': 1745926381.3887439,
  'tool': 'multiply',
  'inputs': {'a': 4, 'b': 5},
  'success': True,
  'result_type': 'int'},
 {'timestamp': 1745926381.3887732,
  'tool': 'get_agent_memory',
  'inputs': {},
  'success': True,
  'result_type': 'dict'}]

# Command-Line Interface

Finally, we'll add a command-line interface to make our agent system accessible from the terminal.

In [ ]:
#| export
def cli_main():
    """Main entry point for the CLI."""
    # Skip argument parsing when run inside a notebook
    in_notebook = 'ipykernel' in sys.modules
    
    if in_notebook:
        # Default behavior for notebook - create an agent and return it
        return Agent()
    
    # Normal CLI operation with argument parsing
    parser = argparse.ArgumentParser(description="CogitareLink CLI")
    parser.add_argument('--version', action='store_true', help='Show version and exit')
    parser.add_argument('--list-tools', action='store_true', help='List available tools')
    parser.add_argument('--run-tool', metavar='TOOL', help='Run a specific tool')
    parser.add_argument('--args', metavar='JSON', help='JSON arguments for tool')
    
    args = parser.parse_args()
    
    # Create default agent
    agent = Agent()
    
    if args.version:
        print(f"CogitareLink version: {agent.run_tool('get_version')}")
        return 0
        
    if args.list_tools:
        tools = agent.run_tool('list_tools')
        print(json.dumps(tools, indent=2))
        return 0
        
    if args.run_tool:
        tool_name = args.run_tool
        tool_args = {}
        
        if args.args:
            try:
                tool_args = json.loads(args.args)
            except json.JSONDecodeError as e:
                print(f"Error parsing arguments: {e}", file=sys.stderr)
                return 1
        
        try:
            result = agent.run_tool(tool_name, **tool_args)
            print(json.dumps(result, indent=2, default=str))
            return 0
        except Exception as e:
            print(f"Error running tool {tool_name}: {e}", file=sys.stderr)
            return 1
    
    # Default behavior - show help
    parser.print_help()
    return 0

In [ ]:
#| export
# Entry point if script is run directly
if __name__ == "__main__":
    sys.exit(cli_main())

usage: ipykernel_launcher.py [-h] [--version] [--list-tools] [--run-tool TOOL]
                             [--args JSON]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/cvardema/Library/Jupyter/runtime/kernel-v3b2852be087291cca03a6ed3cf47cf55f668df364.json


SystemExit: 2

/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Agentic System Prototype

We've established the foundation for an agentic system that can:

1. Register and execute tools
2. Maintain state across multiple tool executions
3. Cache results for improved performance
4. Log actions for transparency
5. Provide a command-line interface

In the next notebook, we'll build on this foundation to create vocabulary-specific tools that leverage CogitareLink's registry, composer, and retriever systems. These tools will allow agents to work with semantic vocabularies like Croissant, handle context composition, and integrate with external resources like Wikidata.